

# convert.py 분석

3DGS의 `convert.py`는 COLMAP CLI를 호출하여 이미지들로부터 카메라 파라미터와 초기 3D 점을 추출합니다.

## COLMAP 설치

**GitHub:** https://github.com/colmap/colmap

**Windows:**
- [Releases](https://github.com/colmap/colmap/releases)에서 pre-built binary 다운로드
- `COLMAP-x.x-windows-cuda.zip` 압축 해제 후 PATH에 추가

**Linux (Ubuntu):**

In [ ]:
sudo apt install colmap



**Conda:**

In [ ]:
conda install -c conda-forge colmap



설치 확인:

In [ ]:
colmap -h



## 개요

```
입력: source_path/input/*.jpg
출력: source_path/sparse/0/ (cameras.bin, images.bin, points3D.bin)
      source_path/images/     (undistorted 이미지)
```



## 명령줄 인자

In [ ]:
parser = ArgumentParser("Colmap converter")
parser.add_argument("--no_gpu", action='store_true')
parser.add_argument("--skip_matching", action='store_true')
parser.add_argument("--source_path", "-s", required=True, type=str)
parser.add_argument("--camera", default="OPENCV", type=str)
parser.add_argument("--colmap_executable", default="", type=str)
parser.add_argument("--resize", action="store_true")
parser.add_argument("--magick_executable", default="", type=str)



| 인자 | 기본값 | 설명 |
|------|--------|------|
| `-s`, `--source_path` | **필수** | 데이터셋 경로 (`input/` 폴더 포함해야 함) |
| `--camera` | `OPENCV` | 카메라 모델 (OPENCV, PINHOLE 등) |
| `--no_gpu` | False | GPU 비활성화 |
| `--skip_matching` | False | Feature extraction/matching 건너뛰기 |
| `--resize` | False | Multi-resolution 이미지 생성 |

## 파이프라인

### Step 1: Feature Extraction

In [ ]:
## Feature extraction
feat_extracton_cmd = colmap_command + " feature_extractor "\
    "--database_path " + args.source_path + "/distorted/database.db \
    --image_path " + args.source_path + "/input \
    --ImageReader.single_camera 1 \
    --ImageReader.camera_model " + args.camera + " \
    --SiftExtraction.use_gpu " + str(use_gpu)
exit_code = os.system(feat_extracton_cmd)



**주요 옵션:**
- `--ImageReader.single_camera 1`: 모든 이미지가 동일한 카메라로 촬영됨
- `--ImageReader.camera_model OPENCV`: 렌즈 왜곡을 고려한 카메라 모델
- `--SiftExtraction.use_gpu 1`: GPU 가속 SIFT 추출

### Step 2: Feature Matching

In [ ]:
## Feature matching
feat_matching_cmd = colmap_command + " exhaustive_matcher \
    --database_path " + args.source_path + "/distorted/database.db \
    --SiftMatching.use_gpu " + str(use_gpu)
exit_code = os.system(feat_matching_cmd)



**Exhaustive Matcher**: 모든 이미지 쌍을 비교합니다 ($N$ 이미지 → $\frac{N(N-1)}{2}$ 쌍).

### Step 3: Mapper (SfM + Bundle Adjustment)

In [ ]:
### Bundle adjustment
# The default Mapper tolerance is unnecessarily large,
# decreasing it speeds up bundle adjustment steps.
mapper_cmd = (colmap_command + " mapper \
    --database_path " + args.source_path + "/distorted/database.db \
    --image_path "  + args.source_path + "/input \
    --output_path "  + args.source_path + "/distorted/sparse \
    --Mapper.ba_global_function_tolerance=0.000001")
exit_code = os.system(mapper_cmd)



**주요 옵션:**
- `--Mapper.ba_global_function_tolerance=0.000001`: Bundle Adjustment 수렴 조건을 엄격하게 설정 (기본값보다 작음 → 속도 향상)

### Step 4: Image Undistortion

In [ ]:
### Image undistortion
## We need to undistort our images into ideal pinhole intrinsics.
img_undist_cmd = (colmap_command + " image_undistorter \
    --image_path " + args.source_path + "/input \
    --input_path " + args.source_path + "/distorted/sparse/0 \
    --output_path " + args.source_path + "\
    --output_type COLMAP")
exit_code = os.system(img_undist_cmd)



**왜 필요한가?** 3DGS 렌더러는 **PINHOLE 카메라만 지원**합니다. OPENCV 모델의 렌즈 왜곡 파라미터(k1, k2, p1, p2)를 적용하여 이상적인 핀홀 이미지로 변환합니다.

### Step 5: Multi-Resolution Images (선택)

`--resize` 옵션 사용 시 ImageMagick으로 다양한 해상도 이미지를 생성합니다.

In [ ]:
if(args.resize):
    print("Copying and resizing...")

    # Resize images.
    os.makedirs(args.source_path + "/images_2", exist_ok=True)
    os.makedirs(args.source_path + "/images_4", exist_ok=True)
    os.makedirs(args.source_path + "/images_8", exist_ok=True)
    # ...
    exit_code = os.system(magick_command + " mogrify -resize 50% " + destination_file)
    # ...
    exit_code = os.system(magick_command + " mogrify -resize 25% " + destination_file)
    # ...
    exit_code = os.system(magick_command + " mogrify -resize 12.5% " + destination_file)



| 폴더 | 해상도 | 용도 |
|------|--------|------|
| `images/` | 100% | 원본 (undistorted) |
| `images_2/` | 50% | `--resolution 2` |
| `images_4/` | 25% | `--resolution 4` |
| `images_8/` | 12.5% | `--resolution 8` |

## 출력 디렉토리 구조

```
source_path/
├── input/              # 원본 이미지 (사용자 제공)
├── distorted/          # COLMAP 작업 디렉토리
│   ├── database.db     # Feature/Match 데이터베이스
│   └── sparse/0/       # 왜곡된 상태의 reconstruction
├── images/             # Undistorted 이미지 (3DGS 학습용)
├── sparse/0/           # 최종 카메라 파라미터
│   ├── cameras.bin     # 카메라 intrinsics
│   ├── images.bin      # 이미지별 pose (extrinsics)
│   └── points3D.bin    # Sparse 3D 점
└── images_2,4,8/       # (--resize 시) Multi-resolution 이미지
```



## 사용 예시

In [ ]:
# 기본 사용
python convert.py -s /path/to/my_scene

# GPU 없이 실행
python convert.py -s /path/to/my_scene --no_gpu

# Multi-resolution 이미지 생성
python convert.py -s /path/to/my_scene --resize

# 이미 매칭 완료된 경우 (undistortion만)
python convert.py -s /path/to/my_scene --skip_matching



---

## 참고 자료

[Computer Vision II: Multiple View Geometry](https://cvg.cit.tum.de/teaching/online/mvg)

[3D Computer Vision | National University of Singapore](https://www.youtube.com/playlist?list=PLxg0CGqViygP47ERvqHw_v7FVnUovJeaz)

[다중관점기하학 개념 정리 - 한국어](https://www.youtube.com/watch?v=ZO91k-zAyMk&list=PLddVNIYmpwlPY6KElbvmfKpcwwVWeggmv)

[컴퓨터비전 특강 - 한국어](https://youtu.be/N6vP0T1Xabg?si=j5Bbx_2YdmYJ5ZwZ)

[고려대학교 Introduction to Computer Vision](https://www.youtube.com/playlist?list=PLW_xDnYtBZJn7cRa7SthQQsXM9s77xrh2)

[CMSC426 Computer Vision Structure From Motion](https://cmsc426.github.io/sfm/)